In [1]:
import pandas as pd
import numpy as np
import h5py
import subprocess
import datetime
import os

from pathlib import Path

from tqdm.notebook import tqdm

from scipy.io import savemat

from mobgap.data import GenericMobilisedDataset
from mobgap.pipeline import MobilisedPipelineImpaired, MobilisedPipelineHealthy

C:\Users\ac4jmi\anaconda3\envs\mobgap_fixed\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\ac4jmi\anaconda3\envs\mobgap_fixed\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\ac4jmi\anaconda3\envs\mobgap_fixed\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.6.1 when using vers

In [2]:
# Experimental Parameters

# which files to find
root_dir = r"X:\insigneo_brc1\GroupFiles\Study_Data\MS_SMART (STH17249; STH18829)\\"
pattern = f"*6MWT.h5"
desired_sensor = "Lumbar"



In [3]:
# Functions
def find_files_with_progress(root_dir, pattern):
    root_dir = Path(root_dir)
    cmd = [
        "powershell",
        "-Command",
        f"Get-ChildItem -Path '{root_dir}' -Recurse -File -Include '{pattern}' | ForEach-Object {{ $_.FullName }}"
    ]

    files = []
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True, bufsize=1)

    pbar = tqdm(desc=f"Scanning {root_dir}", unit="files", dynamic_ncols=True)
    try:
        for line in process.stdout:
            line = line.strip()
            if line:
                files.append(str(Path(line)))
                pbar.update(1)
    finally:
        process.stdout.close()
        process.wait()
        pbar.close()

    print(f"Found {len(files)} matching files.")
    return files

def get_acc_gyr_from_h5(path, desired_sensor, prints = False):
    with h5py.File(path, 'r') as file:
        if prints:
            print()
            attr_names = list(file.attrs.keys())
            print(f"Attribute names: {attr_names}")
            top_level_keys = list(file)
            print(f"    MonitorLabelList: {[x.decode('UTF-8') for x in list(file.attrs["MonitorLabelList"])]}")
            print(f"    CaseIdList: {[x.decode('UTF-8') for x in list(file.attrs["CaseIdList"])]}")

        # get the sensor ID from the label list, assumes element i in MonitorLabelList is the label for element i in CaseIdList.
        desired_sensor_index = [x.decode('UTF-8') for x in list(file.attrs["MonitorLabelList"])].index(desired_sensor)
        desired_sensor_id = list(file.attrs["CaseIdList"])[desired_sensor_index].decode('UTF-8')

        if prints:
            print(f"        Lumbar Sensor ID: {desired_sensor_id}")
            SI_level_keys = list(file[desired_sensor_id])
            cal_level_keys = list(file[desired_sensor_id]["Calibrated"])
            raw_level_keys = list(file[desired_sensor_id]["Raw"])
    
            print("Each SI contains:", SI_level_keys)
            print("    Calibrated contains:", cal_level_keys)
            #print("    Raw contains:", raw_level_keys)

        acc_data = np.array(file[desired_sensor_id]["Calibrated"]["Accelerometers"])
        gyr_data = np.array(file[desired_sensor_id]["Calibrated"]["Gyroscopes"])
        time_data = np.array(file[desired_sensor_id]["Time"])

        if prints:
            print("        Shape of calibrated acc data:", np.shape(acc_data))
            print("        Shape of calibrated gyr data:", np.shape(gyr_data))
            print("        Shape of time", np.shape(time_data))
            print()
    
    if len(acc_data) == len(gyr_data) == len(time_data):
        time_data_reshaped = np.reshape(time_data, (-1, 1))
        acc_gyr_data = np.hstack((time_data_reshaped, acc_data, gyr_data))
        acc_gyr_data_df = pd.DataFrame(data = acc_gyr_data, columns = ["Time", "Acc_X", "Acc_Y", "Acc_Z", "Gyr_X", "Gyr_Y", "Gyr_Z"])

    if prints:
        print(acc_gyr_data_df.head(5))
        print()
        
    return acc_gyr_data_df

def save_to_mat_files(acc_gyr, subject_metadata, path, prints = False):
    data = {"TimeMeasure1": {}}
    
    if prints:
        print(f"NaNs (before interpolation) = {acc_gyr.isna().sum().sum()}")
    
    # get sample rate
    time_seconds = acc_gyr["Time"].astype('int64') / 1e6 # microseconds to seconds
    duration = time_seconds.iloc[-1] - time_seconds.iloc[0]
    if duration <= 0:
        raise ValueError("Invalid or zero-duration time range in acc_gyr.")
    fs = int(round(len(acc_gyr) / duration))

    # Convert timestamps to seconds for MATLAB
    reformatted_time = time_seconds.values.reshape(-1, 1)

    sensors = {
        "LowerBack": {
            "Fs": {"Acc": fs, "Gyr": fs},
            "Acc": acc_gyr[["Acc_X", "Acc_Y", "Acc_Z"]].values,
            "Gyr": acc_gyr[["Gyr_X", "Gyr_Y", "Gyr_Z"]].values,
            "Timestamp": reformatted_time,
        }
    }

    data["TimeMeasure1"]["SixMWT"] = {
        "SU": sensors,
        "StartDateTime": datetime.datetime.fromtimestamp(time_seconds[0]).strftime("%d-%b-%Y %H:%M:%S"),
        "TimeZone": "Europe/UK",
        "TrialName": "Trial 1",
    }

    if prints:
        print(data)
    
    # create infoForAlgo struct using metadata
    infoForAlgo = {
        "TimeMeasure1": {
            "Subject_ID": subject_metadata["IdSubj"].values[0],
            "Cohort": subject_metadata["Patient/Control"].values[0],
            "Gender": subject_metadata["Gender"].values[0],
            "Age": subject_metadata["Age now"].values[0],
            "Weight": 60,
            "Height": 180,
            "SensorHeight": 180*0.575, # estimate but it doens't matter because height is wrong - if we find height this might help
            "SensorType_SU": "OPAL",
            "SensorAttachment_SU": "Body-Worn",
        }
    }

    if prints:
        print(infoForAlgo)

    save_dir = Path(path).parent
    cohort = str(Path(save_dir.parts[-3]))
    if cohort == "Healthy controls":
        cohort_str = "HA"
    elif cohort == "pwMS":
        cohort_str = "MS"
    else:
        print(cohort)
        raise ValueError("Something has gone wrong with the cohort folder names.")
        
    local_copy_path = os.getcwd() + "\\" + cohort_str + "\\" + str(Path(*save_dir.parts[-2:]))
    os.makedirs(local_copy_path, exist_ok=True)

    if prints:
        print(f"Save dir: {local_copy_path}")
    
    # Save combined .mat files
    savemat(os.path.join(local_copy_path, "data.mat"), {"data": data})
    savemat(os.path.join(local_copy_path, "infoForAlgo.mat"), {"infoForAlgo": infoForAlgo})

def get_paths_with_extension(extension):
    paths = []
    dataset_folder = os.path.join(os.getcwd())

    for root, dirs, files in os.walk(dataset_folder):
        for file in files:
            if file.lower().endswith(extension):
                path = os.path.join(root, file)
                paths.append(path)
    
    return paths

In [4]:
# Get all file paths (only needs to be run once so in own cell)
file_paths = find_files_with_progress(root_dir, pattern)
filtered_files = [f for f in file_paths if any(part.startswith("MS") for part in Path(f).parts[5:-1])]
print(f"Filtered down to {len(filtered_files)} files.")
subject_ids = [path.split("\\")[-1].split("_")[0] for path in filtered_files]

Scanning X:\insigneo_brc1\GroupFiles\Study_Data\MS_SMART (STH17249; STH18829): 0files [00:00, ?files/s]

Found 204 matching files.
Filtered down to 163 files.


In [5]:
# get data from h5 files and save as properly-formatted .mat files
# NOTE: these mat files are saved locally - I'm scared of making changes to the drive.

metadata_file_path = root_dir + "Sheffield_MS-SMART_Maindataset.xlsx"
metadata_2_file_path = root_dir + "LongitudinalStudy_MS-SMART.xlsx"
metadata = pd.read_excel(metadata_file_path)
metadata_2 = pd.read_excel(metadata_2_file_path)

for path in filtered_files:
    print(path)
    subject_id = path.split("\\")[-3]
    acc_gyr_data = get_acc_gyr_from_h5(path, desired_sensor)

    if subject_id in metadata_2["IdSubj"].values:
        metadata_row = metadata_2[metadata_2["IdSubj"] == subject_id]
        save_to_mat_files(acc_gyr_data, metadata_row, path, prints = False)
    else:
        print(f"{subject_id} not in metadata")

X:\insigneo_brc1\GroupFiles\Study_Data\MS_SMART (STH17249; STH18829)\Healthy controls\MS019\Baseline_week0\MS019_20151209_133802_OPAL_6MWT.h5
X:\insigneo_brc1\GroupFiles\Study_Data\MS_SMART (STH17249; STH18829)\Healthy controls\MS019\Follow up_week1\MS019_20151218_140304_OPAL_6MWT.h5
X:\insigneo_brc1\GroupFiles\Study_Data\MS_SMART (STH17249; STH18829)\Healthy controls\MS027\Baseline_week0\MS027_20160120_124338_OPAL_6MWT.h5
X:\insigneo_brc1\GroupFiles\Study_Data\MS_SMART (STH17249; STH18829)\Healthy controls\MS027\Follow up_week1\MS027_20160127_111414_OPAL_6MWT.h5
X:\insigneo_brc1\GroupFiles\Study_Data\MS_SMART (STH17249; STH18829)\Healthy controls\MS051\Baseline_week0\MS051_20160419_122500_OPAL_6MWT.h5
X:\insigneo_brc1\GroupFiles\Study_Data\MS_SMART (STH17249; STH18829)\Healthy controls\MS051\Follow up_week1\MS051_20160427_114936_OPAL_6MWT.h5
X:\insigneo_brc1\GroupFiles\Study_Data\MS_SMART (STH17249; STH18829)\Healthy controls\MS052\Baseline_week0\MS052_20160427_142653_OPAL_6MWT.h5
X:\

In [6]:
# build a dataset from the .mat files

mat_files = get_paths_with_extension(".mat")
mat_files = [file for file in mat_files if file.endswith("data.mat")]

dataset = GenericMobilisedDataset(
    paths_list = mat_files,
    test_level_names = ["TimeMeasure", "Test"], # This is the structure (without numbers) of the data inside the data.mat struct.
    measurement_condition = "laboratory",
    parent_folders_as_metadata=["cohort", "subject", "timepoint"] # these are the column headings for the metadata, the columns get populated with the folder names.
)

display(dataset)

C:\Users\ac4jmi\anaconda3\envs\mobgap_fixed\Lib\site-packages\mobgap\data\_mobilised_matlab_loader.py:1245: UserWarning: Global caching is a little tricky to get right and our implementation is not yet battle-tested. Please double check that the results are correct and report any issues you find.
  return hybrid_cache(self.memory, 1)(_load_test_data_without_checks)(


,cohort,subject,timepoint,TimeMeasure,Test
0,HA,MS019,Baseline_week0,TimeMeasure1,SixMWT
1,HA,MS019,Follow up_week1,TimeMeasure1,SixMWT
2,HA,MS027,Baseline_week0,TimeMeasure1,SixMWT
3,HA,MS027,Follow up_week1,TimeMeasure1,SixMWT
4,HA,MS051,Baseline_week0,TimeMeasure1,SixMWT
...,...,...,...,...,...
129,MS,MS066,Follow up_week48,TimeMeasure1,SixMWT
130,MS,MS067,Baseline_week0,TimeMeasure1,SixMWT
131,MS,MS067,Follow up_week24,TimeMeasure1,SixMWT
132,MS,MS067,Follow up_week48,TimeMeasure1,SixMWT


In [8]:
# run the appropriate pipeline on each cohort and save as csv files.

healthy_pipeline = MobilisedPipelineHealthy()
impaired_pipeline = MobilisedPipelineImpaired()

cohorts = ["HA", "MS"]

all_pipeline_wb_params = pd.DataFrame()
all_pipeline_agg_params = pd.DataFrame()

for cohort in cohorts:
    subjects = list({row[1] for row in dataset.group_labels if row[0] == cohort})
    for subject in subjects:
        timepoints = list(row[2] for row in dataset.group_labels if (row[0] == cohort and row[1] == subject))
        for timepoint in timepoints:
            data = dataset.get_subset(cohort=cohort, subject=subject, timepoint=timepoint)
            if cohort == "HA":
                pipeline = healthy_pipeline.safe_run(data)
            else:
                pipeline = impaired_pipeline.safe_run(data)

            # insert some demographic details so we know who is who when saving
            pipeline.per_wb_parameters_.insert(loc = 0, column = "subject_id", value = str(subject))
            pipeline.aggregated_parameters_.insert(loc = 0, column = "subject_id", value = str(subject))
            pipeline.per_wb_parameters_.insert(loc = 1, column = "cohort", value = cohort)
            pipeline.aggregated_parameters_.insert(loc = 1, column = "cohort", value = cohort)
            pipeline.per_wb_parameters_.insert(loc = 2, column = "timepoint", value = timepoint)
            pipeline.aggregated_parameters_.insert(loc = 2, column = "timepoint", value = timepoint)

        
            all_pipeline_wb_params = pd.concat([all_pipeline_wb_params, pipeline.per_wb_parameters_]).reset_index(drop=True)
            all_pipeline_agg_params = pd.concat([all_pipeline_agg_params, pipeline.aggregated_parameters_]).reset_index(drop=True)

print("Pipeline finished running!")

display(all_pipeline_agg_params)

# drop spatial parameters
drop_regex = "stride_length|walking_speed"
all_pipeline_wb_params = all_pipeline_wb_params.drop(columns=all_pipeline_wb_params.filter(regex=drop_regex).columns)
all_pipeline_agg_params = all_pipeline_agg_params.drop(columns=all_pipeline_agg_params.filter(regex=drop_regex).columns)

# save as csv
all_pipeline_wb_params.to_csv(r"MS_SMART_per_wb_temporal_parameters.csv")
all_pipeline_agg_params.to_csv(r"MS_SMART_per_trial_temporal_parameters.csv")


Pipeline finished running!


,subject_id,cohort,timepoint,wb_all__count,total_walking_duration_min,wb_all__n_raw_initial_contacts__sum,wb_all__duration_s__avg,wb_all__duration_s__p90,wb_all__duration_s__var,wb_all__cadence_spm__avg,...,wb_30__count,wb_30__walking_speed_mps__avg,wb_30__stride_length_m__avg,wb_30__cadence_spm__avg,wb_30__stride_duration_s__avg,wb_30__walking_speed_mps__p90,wb_30__cadence_spm__p90,wb_30__walking_speed_mps__var,wb_30__stride_length_m__var,wb_60__count
0,MS011,MS,Baseline_week0,1,4.190365,394,251.421875,251.421875,NaN,95.332749,...,1,NaN,NaN,95.332749,0.654744,NaN,95.332749,NaN,NaN,1
1,MS011,MS,Follow up_week24,1,6.025391,569,361.523438,361.523438,NaN,96.047368,...,1,NaN,NaN,96.047368,0.815284,NaN,96.047368,NaN,NaN,1
2,MS011,MS,Follow up_week96,1,5.974479,543,358.46875,358.46875,NaN,91.561141,...,1,NaN,NaN,91.561141,0.74471,NaN,91.561141,NaN,NaN,1
3,MS045,MS,Baseline_week0,1,5.797917,664,347.875,347.875,NaN,115.241429,...,1,NaN,NaN,115.241429,0.536299,NaN,115.241429,NaN,NaN,1
4,MS045,MS,Follow up_week24,1,6.076562,718,364.59375,364.59375,NaN,118.250021,...,1,NaN,NaN,118.250021,0.523627,NaN,118.250021,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102,MS066,MS,Follow up_week48,1,5.980859,577,358.851562,358.851562,NaN,96.332085,...,1,NaN,NaN,96.332085,0.626268,NaN,96.332085,NaN,NaN,1
103,MS067,MS,Baseline_week0,1,1.320833,137,79.25,79.25,NaN,94.930467,...,1,2.030558,2.573473,94.930467,0.587037,2.030558,94.930467,NaN,NaN,1
104,MS067,MS,Follow up_week24,1,2.501302,197,150.078125,150.078125,NaN,82.067698,...,1,1.406721,2.050233,82.067698,0.769631,1.406721,82.067698,NaN,NaN,1
105,MS067,MS,Follow up_week48,1,1.197917,109,71.875,71.875,NaN,95.452232,...,1,1.903864,2.390950,95.452232,0.665509,1.903864,95.452232,NaN,NaN,1
